# 17.1 Setup & Imports

### Objective

Initialize the Customer Geographic Intelligence environment for Workbook 17.

This workbook extends the Customer Activation Master created in Workbook 16 by introducing a reusable geographic intelligence layer. Throughout this workbook, customer addresses will be transformed into geographic insights that support neighborhood marketing, ZIP code analysis, and future distribution planning.

In [243]:
from pathlib import Path
import sys

import duckdb
import numpy as np
import pandas as pd

In [244]:
# Import project configuration

sys.path.append(str(Path.cwd().parent))

from config import GOOGLE_MAPS_API_KEY

In [245]:
# Display settings

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

In [246]:
# Project directories

BASE_DIR = Path.cwd().parent

DATA_DIR = BASE_DIR / "data"

RAW_DIR = DATA_DIR / "raw"
CLEANED_DIR = DATA_DIR / "cleaned"
PROCESSED_DIR = DATA_DIR / "processed"

CUSTOMER_DIR = CLEANED_DIR / "customer"
GEOGRAPHY_DIR = CLEANED_DIR / "geography"

GEOGRAPHY_DIR.mkdir(parents=True, exist_ok=True)

In [251]:
# File paths

# File paths

CUSTOMER_ACTIVATION_MASTER_PATH = CUSTOMER_DIR / "customer_activation_master.csv"

UNIQUE_ADDRESSES_PATH = GEOGRAPHY_DIR / "unique_customer_addresses.csv"
GEOCODE_LOOKUP_PATH = GEOGRAPHY_DIR / "customer_geocode_lookup.csv"
CUSTOMER_GEOGRAPHY_MASTER_PATH = GEOGRAPHY_DIR / "customer_geography_master.csv"
ZIP_SUMMARY_PATH = GEOGRAPHY_DIR / "zip_summary.csv"
CITY_SUMMARY_PATH = (GEOGRAPHY_DIR / "city_summary.csv")

# 17.2 Import Customer Activation Master

### Objective

Import the Customer Activation Master created in Workbook 16. This dataset serves as the foundation for the geographic intelligence workflow by combining customer information, marketing engagement, and standardized address fields into a single analysis-ready table.

In [252]:
customer_activation_master = pd.read_csv(CUSTOMER_ACTIVATION_MASTER_PATH)

customer_activation_master.head()

,customer_id_clean,phone_number,phone_clean,address_raw,address_clean_customer,street_number_customer,street_name_customer,street_suffix_customer,unit_customer,address_join_key_customer,campaign_touch_count,first_campaign_seen,last_campaign_seen,customer_reply_count,verified_response_any,stop_any,address_join_key_sms,activation_status,eligible_for_future_campaign
0,1,9162129352,9162129352,6710 37 Ave,6710 37 AVE,"6,710.00",37TH,AVE,NaN,6710_37TH_AVE,1,D1_A,D1_A,0.00,False,False,6710_37TH_AVE,NO_RESPONSE,True
1,4,9162301496,9162301496,6425 Somis Way,6425 SOMIS WAY,"6,425.00",SOMIS,WAY,NaN,6425_SOMIS_WAY,2,D3,D6,0.00,False,False,6425_SOMIS_WAY,NO_RESPONSE,True
2,5,9163292467,9163292467,6100 48th Ave #2202,6100 48TH AVE #2202,"6,100.00",48TH,AVE,#2202,6100_48TH_AVE,2,D3,D3_2,0.00,False,False,6100_48TH_AVE,NO_RESPONSE,True
3,6,2792720793,2792720793,3826 25th Ave,3826 25TH AVE,"3,826.00",25TH,AVE,NaN,3826_25TH_AVE,1,D1_A,D1_A,0.00,False,False,3826_25TH_AVE,NO_RESPONSE,True
4,7,9162065582,9162065582,7101 Gerber Rd #216 #8106,7101 GERBER RD #216 #8106,"7,101.00",GERBER,RD,#216 #8106,7101_GERBER_RD,2,D3,D6,0.00,False,False,7101_GERBER_RD,NO_RESPONSE,True


# 17.3 Review Geographic Fields

### Objective

Review the address fields available in the Customer Activation Master.

This step confirms that the workbook has the required customer address fields needed to build a reusable geographic lookup table. Before geocoding or mapping, the address structure must be verified so each customer can be connected to a standardized geographic record.

In [253]:
geographic_fields = [
    "customer_id_clean",
    "phone_clean",
    "address_raw",
    "address_join_key_customer",
    "campaign_touch_count",
    "activation_status",
    "eligible_for_future_campaign",
]

customer_activation_master[geographic_fields].head()

,customer_id_clean,phone_clean,address_raw,address_join_key_customer,campaign_touch_count,activation_status,eligible_for_future_campaign
0,1,9162129352,6710 37 Ave,6710_37TH_AVE,1,NO_RESPONSE,True
1,4,9162301496,6425 Somis Way,6425_SOMIS_WAY,2,NO_RESPONSE,True
2,5,9163292467,6100 48th Ave #2202,6100_48TH_AVE,2,NO_RESPONSE,True
3,6,2792720793,3826 25th Ave,3826_25TH_AVE,1,NO_RESPONSE,True
4,7,9162065582,7101 Gerber Rd #216 #8106,7101_GERBER_RD,2,NO_RESPONSE,True


In [254]:
customer_activation_master[geographic_fields].isna().sum()

customer_id_clean               0
phone_clean                     0
address_raw                     0
address_join_key_customer       0
campaign_touch_count            0
activation_status               0
eligible_for_future_campaign    0
dtype: int64

In [255]:
customer_activation_master["address_join_key_customer"].nunique()

5316

# 17.4 Build Unique Address Table

### Objective

Create a reusable table containing one record for each standardized customer address.

Rather than geocoding every customer record, Workbook 17 geocodes each unique address once and stores the results in a reusable lookup table. This approach improves efficiency, reduces duplicate processing, and creates a scalable geographic foundation for future analyses.

In [256]:
unique_addresses = (
    customer_activation_master[
        [
            "address_join_key_customer",
            "address_raw",
        ]
    ]
    .sort_values(["address_join_key_customer", "address_raw"])
    .drop_duplicates(subset="address_join_key_customer")
    .reset_index(drop=True)
)

unique_addresses.head()

,address_join_key_customer,address_raw
0,"""2315_STOCKTON_BLVD","""2315 Stockton Blvd (Emergency, In Back)"""
1,"""2394_GLEN_ELLEN_CIR","""2394 Glen Ellen Cir ,#2"""
2,"""3804_40_AVE","""3804 40 Ave ,#9"""
3,"""6163_44_ST,_HOUSE_ON_THE_LEFT""","""6163 44 St, House On The Left"""
4,.,.


In [257]:
print(f"Unique standardized addresses: {len(unique_addresses):,}")

Unique standardized addresses: 5,316


In [258]:
unique_addresses.to_csv(UNIQUE_ADDRESSES_PATH, index=False)

print(f"Saved: {UNIQUE_ADDRESSES_PATH.name}")

Saved: unique_customer_addresses.csv


# 17.5 Geographic Data Readiness QA

### Objective

Validate the unique address table before geographic enrichment.

This step checks whether standardized address records are complete, unique, and ready to become the reusable geocode lookup table for Workbook 17.

In [259]:
unique_address_qa = pd.DataFrame(
    {
        "metric": [
            "Total Unique Address Records",
            "Missing Address Join Keys",
            "Missing Raw Addresses",
            "Address Normalization Collisions",
        ],
        "value": [
            len(unique_addresses),
            unique_addresses["address_join_key_customer"].isna().sum(),
            unique_addresses["address_raw"].isna().sum(),
            unique_addresses["address_join_key_customer"].duplicated().sum(),
        ],
    }
)

unique_address_qa

,metric,value
0,Total Unique Address Records,5316
1,Missing Address Join Keys,0
2,Missing Raw Addresses,0
3,Address Normalization Collisions,0


In [260]:
normalization_collisions = unique_addresses[
    unique_addresses["address_join_key_customer"].duplicated(keep=False)
].sort_values("address_join_key_customer")

normalization_collisions.head(20)

,address_join_key_customer,address_raw


# 17.6 Prepare Geocode Lookup Table

## Objective

Prepare the reusable customer geocode lookup table.

This step creates one record for each standardized customer address and initializes the geographic fields required for enrichment. Rather than performing geocoding inside the notebook, Workbook 17 exports the lookup table and uses a reusable Google Maps Geocoding script to enrich each unique address.

The lookup table is keyed by `address_join_key_customer`, allowing each standardized address to be geocoded once and reused across customer, ZIP code, city, carrier route, and future geographic marketing analyses.

**Workflow**

Customer Activation Master

↓

Unique Customer Addresses

↓

Prepare Geocode Lookup Table *(Workbook 17.6)*

↓

Run `scripts/geocode_lookup.py`

↓

Completed Customer Geocode Lookup

↓

Workbook 17.7 Import Completed Geocode Lookup

In [261]:
geocode_lookup = unique_addresses.copy()

geocode_lookup["city"] = pd.NA
geocode_lookup["state"] = pd.NA
geocode_lookup["zip_code"] = pd.NA
geocode_lookup["latitude"] = np.nan
geocode_lookup["longitude"] = np.nan
geocode_lookup["geocode_status"] = "PENDING"

geocode_lookup.head()

,address_join_key_customer,address_raw,city,state,zip_code,latitude,longitude,geocode_status
0,"""2315_STOCKTON_BLVD","""2315 Stockton Blvd (Emergency, In Back)""",<NA>,<NA>,<NA>,NaN,NaN,PENDING
1,"""2394_GLEN_ELLEN_CIR","""2394 Glen Ellen Cir ,#2""",<NA>,<NA>,<NA>,NaN,NaN,PENDING
2,"""3804_40_AVE","""3804 40 Ave ,#9""",<NA>,<NA>,<NA>,NaN,NaN,PENDING
3,"""6163_44_ST,_HOUSE_ON_THE_LEFT""","""6163 44 St, House On The Left""",<NA>,<NA>,<NA>,NaN,NaN,PENDING
4,.,.,<NA>,<NA>,<NA>,NaN,NaN,PENDING


In [262]:
geocode_lookup["geocode_status"].value_counts()

geocode_status
PENDING    5316
Name: count, dtype: int64

# 17.7 Import Completed Geocode Lookup

## Objective

Import the completed customer geocode lookup table.

This section loads the geographic enrichment generated by the reusable Google Maps Geocoding pipeline. Each standardized customer address now includes geographic attributes such as formatted address, city, state, ZIP code, latitude, longitude, and geocode status.

The completed lookup table becomes the geographic dimension used throughout the remainder of Workbook 17 for customer geography, ZIP code analytics, city-level analysis, and future geographic marketing optimization.

In [263]:
completed_geocode_lookup = pd.read_csv(GEOCODE_LOOKUP_PATH)

completed_geocode_lookup["zip_code"] = (
    completed_geocode_lookup["zip_code"]
    .astype("string")
    .str.extract(r"(\d{5})")
)

completed_geocode_lookup.head()

,address_join_key_customer,address_raw,formatted_address,city,state,zip_code,country,latitude,longitude,geocode_status
0,"""2315_STOCKTON_BLVD","""2315 Stockton Blvd (Emergency, In Back)""","2315 Stockton Blvd, Sacramento, CA 958172201, USA",Sacramento,CA,95817,United States,38.55,-121.46,OK
1,"""2394_GLEN_ELLEN_CIR","""2394 Glen Ellen Cir ,#2""","2394 Glen Ellen Cir #2, Sacramento, CA 95822, USA",Sacramento,CA,95822,United States,38.52,-121.48,OK
2,"""3804_40_AVE","""3804 40 Ave ,#9""","3804 40th Ave #9, Sacramento, CA 95824, USA",Sacramento,CA,95824,United States,38.52,-121.47,OK
3,"""6163_44_ST,_HOUSE_ON_THE_LEFT""","""6163 44 St, House On The Left""","6163 44th St, Sacramento, CA 95824, USA",Sacramento,CA,95824,United States,38.51,-121.46,OK
4,.,.,"Sacramento, CA, USA",Sacramento,CA,<NA>,United States,38.58,-121.49,OK


In [264]:
completed_geocode_lookup["geocode_status"].value_counts()

geocode_status
OK               5314
REQUEST_ERROR       2
Name: count, dtype: int64

In [265]:
print(f"Completed geocode records: {len(completed_geocode_lookup):,}")
print(
    f"Successfully geocoded: "
    f"{(completed_geocode_lookup['geocode_status'] == 'OK').sum():,}"
)

Completed geocode records: 5,316
Successfully geocoded: 5,314


In [266]:
completed_geocode_lookup.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5316 entries, 0 to 5315
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   address_join_key_customer  5316 non-null   object 
 1   address_raw                5316 non-null   object 
 2   formatted_address          5314 non-null   object 
 3   city                       5312 non-null   object 
 4   state                      5314 non-null   object 
 5   zip_code                   5265 non-null   string 
 6   country                    5314 non-null   object 
 7   latitude                   5314 non-null   float64
 8   longitude                  5314 non-null   float64
 9   geocode_status             5316 non-null   object 
dtypes: float64(2), object(7), string(1)
memory usage: 415.4+ KB


In [267]:
request_errors = completed_geocode_lookup[
    completed_geocode_lookup["geocode_status"] != "OK"
]

request_errors

,address_join_key_customer,address_raw,formatted_address,city,state,zip_code,country,latitude,longitude,geocode_status
659,3390_16TH_AVE,3390 16th Ave,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,REQUEST_ERROR
2237,4931_STOCKTON_BLVD,4931 Stockton Blvd,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,REQUEST_ERROR


# 17.8 SQL Customer Geography Master

## Objective

Build the Customer Geography Master table.

This section joins the completed Google Maps geocode lookup table with the Customer Activation Master from Workbook 16. The resulting dataset combines customer activation metrics, SMS engagement, marketing eligibility, and geographic attributes into one reusable customer geography table.

In [268]:
customer_geography_master = duckdb.sql(
    """
    SELECT

        c.*,

        g.city,
        g.state,
        g.zip_code,
        g.latitude,
        g.longitude,
        g.geocode_status

    FROM customer_activation_master c

    LEFT JOIN completed_geocode_lookup g

    ON c.address_join_key_customer = g.address_join_key_customer

    """
).df()

customer_geography_master["street_number_customer"] = (
    customer_geography_master["street_number_customer"]
    .astype("Int64")
)

customer_geography_master["zip_code"] = (
    customer_geography_master["zip_code"]
    .fillna("")
    .astype(str)
    .str.extract(r"(\d{5})")
)

customer_geography_master.head()

,customer_id_clean,phone_number,phone_clean,address_raw,address_clean_customer,street_number_customer,street_name_customer,street_suffix_customer,unit_customer,address_join_key_customer,campaign_touch_count,first_campaign_seen,last_campaign_seen,customer_reply_count,verified_response_any,stop_any,address_join_key_sms,activation_status,eligible_for_future_campaign,city,state,zip_code,latitude,longitude,geocode_status
0,1,9162129352,9162129352,6710 37 Ave,6710 37 AVE,6710,37TH,AVE,None,6710_37TH_AVE,1,D1_A,D1_A,0.00,False,False,6710_37TH_AVE,NO_RESPONSE,True,Sacramento,CA,95824,38.52,-121.42,OK
1,4,9162301496,9162301496,6425 Somis Way,6425 SOMIS WAY,6425,SOMIS,WAY,None,6425_SOMIS_WAY,2,D3,D6,0.00,False,False,6425_SOMIS_WAY,NO_RESPONSE,True,Sacramento,CA,95828,38.51,-121.41,OK
2,5,9163292467,9163292467,6100 48th Ave #2202,6100 48TH AVE #2202,6100,48TH,AVE,#2202,6100_48TH_AVE,2,D3,D3_2,0.00,False,False,6100_48TH_AVE,NO_RESPONSE,True,Sacramento,CA,95828,38.51,-121.43,OK
3,6,2792720793,2792720793,3826 25th Ave,3826 25TH AVE,3826,25TH,AVE,None,3826_25TH_AVE,1,D1_A,D1_A,0.00,False,False,3826_25TH_AVE,NO_RESPONSE,True,Sacramento,CA,95820,38.53,-121.47,OK
4,7,9162065582,9162065582,7101 Gerber Rd #216 #8106,7101 GERBER RD #216 #8106,7101,GERBER,RD,#216 #8106,7101_GERBER_RD,2,D3,D6,0.00,False,False,7101_GERBER_RD,NO_RESPONSE,True,Sacramento,CA,95828,38.48,-121.42,OK


In [269]:
print(f"Customer Geography Master records: {len(customer_geography_master):,}")

print(
    f"Successfully geocoded customers: "
    f"{(customer_geography_master['geocode_status'] == 'OK').sum():,}"
)

Customer Geography Master records: 9,557
Successfully geocoded customers: 9,555


In [270]:
customer_geography_master["geocode_status"].value_counts()

geocode_status
OK               9555
REQUEST_ERROR       2
Name: count, dtype: int64

In [271]:
customer_geography_master.to_csv(
    CUSTOMER_GEOGRAPHY_MASTER_PATH,
    index=False,
)

print(f"Saved: {CUSTOMER_GEOGRAPHY_MASTER_PATH.name}")

Saved: customer_geography_master.csv


In [272]:
customer_geography_master[
    [
        "address_raw",
        "city",
        "state",
        "zip_code",
        "latitude",
        "longitude",
        "geocode_status",
    ]
].sample(10)

,address_raw,city,state,zip_code,latitude,longitude,geocode_status
5130,5660 62nd St,Sacramento,CA,95824,38.52,-121.43,OK
9525,4609 38th Ave,Sacramento,CA,95824,38.52,-121.45,OK
6891,6650 Nielsen Way,Sacramento,CA,95820,38.54,-121.43,OK
1317,6624 Lemonhill Ave,Sacramento,CA,95824,38.52,-121.43,OK
4684,4815 Franklin Blvd,Sacramento,CA,95820,38.53,-121.47,OK
4325,5021 Mendocino Blvd,Sacramento,CA,95820,38.53,-121.46,OK
1108,7401 Norbeck Way,Sacramento,CA,95824,38.51,-121.42,OK
29,4094 16 Ave,Sacramento,CA,95820,38.54,-121.46,OK
4907,6507 Wesely Avenue A,Sacramento,CA,95823,38.51,-121.46,OK
9345,6008 69 St,Sacramento,CA,95824,38.52,-121.42,OK


# 17.9 ZIP Code Distribution Summary

## Objective

Summarize the geographic distribution of customers by ZIP code.

This section aggregates the Customer Geography Master by ZIP code to identify where customers are concentrated throughout the market area. The resulting summary provides the foundation for geographic marketing analysis, direct mail planning, territory prioritization, and future Tableau mapping.

In [273]:
zip_summary = duckdb.sql(
    """
    SELECT

        zip_code,

        COUNT(*) AS customer_count,

        SUM(
            CASE
                WHEN verified_response_any THEN 1
                ELSE 0
            END
        ) AS verified_customers,

        SUM(
            CASE
                WHEN eligible_for_future_campaign THEN 1
                ELSE 0
            END
        ) AS eligible_customers

    FROM customer_geography_master

    WHERE zip_code IS NOT NULL

    GROUP BY zip_code

    ORDER BY customer_count DESC

    """
).df()

zip_summary["verified_customers"] = (
    zip_summary["verified_customers"]
    .astype(int)
)

zip_summary["eligible_customers"] = (
    zip_summary["eligible_customers"]
    .astype(int)
)

zip_summary["verified_response_rate"] = (
    zip_summary["verified_customers"]
    / zip_summary["customer_count"]
).round(4)

zip_summary.head(20)

,zip_code,customer_count,verified_customers,eligible_customers,verified_response_rate
0,95820,3303,158,2923,0.05
1,95824,2895,125,2613,0.04
2,95823,1209,51,1087,0.04
3,95828,798,48,697,0.06
4,95822,532,31,480,0.06
5,95817,499,20,452,0.04
6,95826,83,3,73,0.04
7,95816,45,2,41,0.04
8,95818,40,2,35,0.05
9,95819,36,1,33,0.03


In [274]:
print(f"ZIP codes represented: {len(zip_summary):,}")
print(f"Customers with ZIP codes: {zip_summary['customer_count'].sum():,}")

ZIP codes represented: 29
Customers with ZIP codes: 9,501


In [275]:
zip_summary.describe(include="all")

,zip_code,customer_count,verified_customers,eligible_customers,verified_response_rate
count,29,29.00,29.00,29.00,29.00
unique,29,NaN,NaN,NaN,NaN
top,95820,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,NaN
mean,NaN,327.62,15.24,292.72,0.02
std,NaN,819.86,37.86,731.89,0.05
min,NaN,1.00,0.00,1.00,0.00
25%,NaN,2.00,0.00,1.00,0.00
50%,NaN,5.00,0.00,4.00,0.00
75%,NaN,45.00,2.00,41.00,0.04


In [276]:
zip_summary.sort_values(
    "verified_response_rate",
    ascending=False,
).head(10)

,zip_code,customer_count,verified_customers,eligible_customers,verified_response_rate
16,95827,4,1,1,0.25
3,95828,798,48,697,0.06
4,95822,532,31,480,0.06
8,95818,40,2,35,0.05
0,95820,3303,158,2923,0.05
7,95816,45,2,41,0.04
1,95824,2895,125,2613,0.04
2,95823,1209,51,1087,0.04
5,95817,499,20,452,0.04
6,95826,83,3,73,0.04


In [277]:
zip_summary.to_csv(
    ZIP_SUMMARY_PATH,
    index=False,
)

print(f"Saved: {ZIP_SUMMARY_PATH.name}")

Saved: zip_summary.csv


# 17.10 City Distribution Summary

## Objective

Summarize the geographic distribution of customers by city.

This section aggregates the Customer Geography Master by city to identify where customers are concentrated throughout the service area. The resulting summary supports city-level market analysis, regional marketing opportunities, delivery expansion evaluation, and future Tableau geographic dashboards.

In [278]:
city_summary = duckdb.sql(
    """
    SELECT

        city,

        COUNT(*) AS customer_count,

        SUM(
            CASE
                WHEN verified_response_any THEN 1
                ELSE 0
            END
        ) AS verified_customers,

        SUM(
            CASE
                WHEN eligible_for_future_campaign THEN 1
                ELSE 0
            END
        ) AS eligible_customers

    FROM customer_geography_master

    WHERE city IS NOT NULL

    GROUP BY city

    ORDER BY customer_count DESC

    """
).df()

city_summary["verified_customers"] = (
    city_summary["verified_customers"]
    .astype(int)
)

city_summary["eligible_customers"] = (
    city_summary["eligible_customers"]
    .astype(int)
)

city_summary["verified_response_rate"] = (
    city_summary["verified_customers"]
    / city_summary["customer_count"]
).round(4)

city_summary.head(20)

,city,customer_count,verified_customers,eligible_customers,verified_response_rate
0,Sacramento,9533,445,8515,0.05
1,Arden-Arcade,5,0,5,0.00
2,Elk Grove,5,0,5,0.00
3,Rancho Cordova,3,0,2,0.00
4,Rio Linda,3,0,3,0.00
5,North Highlands,1,0,1,0.00
6,Vallejo,1,0,1,0.00


In [279]:
print(f"Cities represented: {len(city_summary):,}")
print(f"Customers with city assigned: {city_summary['customer_count'].sum():,}")

Cities represented: 7
Customers with city assigned: 9,551


In [280]:
city_summary.describe(include="all")

,city,customer_count,verified_customers,eligible_customers,verified_response_rate
count,7,7.00,7.00,7.00,7.00
unique,7,NaN,NaN,NaN,NaN
top,Sacramento,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,NaN
mean,NaN,"1,364.43",63.57,"1,218.86",0.01
std,NaN,"3,602.00",168.19,"3,217.30",0.02
min,NaN,1.00,0.00,1.00,0.00
25%,NaN,2.00,0.00,1.50,0.00
50%,NaN,3.00,0.00,3.00,0.00
75%,NaN,5.00,0.00,5.00,0.00


In [281]:
city_summary.sort_values(
    "verified_response_rate",
    ascending=False,
).head(10)

,city,customer_count,verified_customers,eligible_customers,verified_response_rate
0,Sacramento,9533,445,8515,0.05
1,Arden-Arcade,5,0,5,0.00
2,Elk Grove,5,0,5,0.00
3,Rancho Cordova,3,0,2,0.00
4,Rio Linda,3,0,3,0.00
5,North Highlands,1,0,1,0.00
6,Vallejo,1,0,1,0.00
